# チュートリアル5: 評価と解析

このチュートリアルでは、生成された分子の包括的な評価と解析方法を学びます。

**所要時間**: 25分

**学習内容**:
- 安定性メトリクスの詳細計算（原子・分子レベル）
- 許容結合数の厳密な適用
- RDKit統合による化学的妥当性検証
- SMILES重複除去による一意性評価
- 新規性アルゴリズム
- プロパティ分布解析
- 包括的評価パイプライン

**前提知識**: チュートリアル1（基本的な分子生成）

**重要**: フォールバックなしの厳密な評価を実施します。


## セットアップとインポート


In [ ]:
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from qm9.analyze import check_stability, analyze_stability_for_molecules
from qm9 import utils as qm9_utils

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {device}')
print(f'RDKit バージョン: {Chem.rdBase.rdkitVersion}')


## 1. 安定性メトリクス

分子の化学的安定性を評価する基本的なメトリクスです。

### 原子レベルの安定性

各原子タイプに対して、許容される結合数の範囲が定義されています:

| 原子 | 許容結合数 |
|------|------------|
| H    | 0, 1       |
| C    | 3, 4       |
| N    | 2, 3       |
| O    | 1, 2       |
| F    | 0, 1       |

**重要**: これらの制約は厳密に適用され、例外は許可されません。


In [ ]:
# 許容結合数の定義（QM9データセット用）
ALLOWED_BONDS = {
    'H': [0, 1],
    'C': [3, 4],
    'N': [2, 3],
    'O': [1, 2],
    'F': [0, 1],
}

# 原子番号から記号へのマッピング
ATOM_DECODER = ['H', 'C', 'N', 'O', 'F']

print('許容結合数の定義:')
for atom, bonds in ALLOWED_BONDS.items():
    print(f'  {atom}: {bonds}')


## 2. 結合数の計算

原子間距離に基づいて結合を判定し、各原子の結合数を計算します。

**距離閾値**:
- 結合距離: 原子タイプの組み合わせによって異なる
- 典型的な値: C-C 1.7Å, C-H 1.2Å, など


In [ ]:
def compute_bond_counts(positions, atom_types, dataset_info):
    """
    各原子の結合数を計算（厳密な距離閾値使用）
    
    Parameters:
    -----------
    positions : torch.Tensor [n_atoms, 3]
        原子座標
    atom_types : torch.Tensor [n_atoms]
        原子タイプ（インデックス）
    dataset_info : dict
        データセット情報（結合距離閾値を含む）
    
    Returns:
    --------
    bond_counts : np.ndarray [n_atoms]
        各原子の結合数
    """
    n_atoms = len(positions)
    bond_counts = np.zeros(n_atoms, dtype=int)
    
    # 距離行列の計算
    distances = torch.cdist(positions.unsqueeze(0), positions.unsqueeze(0))[0]
    
    # 結合距離マトリックス
    bonds_dict = dataset_info['bonds']
    
    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            atom_i = ATOM_DECODER[atom_types[i]]
            atom_j = ATOM_DECODER[atom_types[j]]
            
            # 結合距離閾値の取得
            if (atom_i, atom_j) in bonds_dict:
                threshold = bonds_dict[(atom_i, atom_j)]
            elif (atom_j, atom_i) in bonds_dict:
                threshold = bonds_dict[(atom_j, atom_i)]
            else:
                # 未定義の組み合わせはスキップ
                continue
            
            # 結合判定
            if distances[i, j] < threshold:
                bond_counts[i] += 1
                bond_counts[j] += 1
    
    return bond_counts

print('結合数計算関数を定義しました。')


## 3. 原子安定性のチェック

各原子について、結合数が許容範囲内かをチェックします。


In [ ]:
def check_atom_stability(positions, atom_types, charges, dataset_info):
    """
    原子レベルの安定性チェック（厳密）
    
    Parameters:
    -----------
    positions : torch.Tensor [n_atoms, 3]
        原子座標
    atom_types : torch.Tensor [n_atoms]
        原子タイプ
    charges : torch.Tensor [n_atoms, 1]
        電荷
    dataset_info : dict
        データセット情報
    
    Returns:
    --------
    n_stable : int
        安定な原子の数
    stable_mask : np.ndarray [n_atoms]
        各原子が安定かどうか
    """
    n_atoms = len(positions)
    
    # 結合数の計算
    bond_counts = compute_bond_counts(positions, atom_types, dataset_info)
    
    # 安定性のチェック
    stable_mask = np.zeros(n_atoms, dtype=bool)
    
    for i in range(n_atoms):
        atom_symbol = ATOM_DECODER[atom_types[i]]
        bond_count = bond_counts[i]
        
        # 許容結合数のチェック
        if bond_count in ALLOWED_BONDS[atom_symbol]:
            stable_mask[i] = True
    
    n_stable = stable_mask.sum()
    
    return n_stable, stable_mask

print('原子安定性チェック関数を定義しました。')


## 4. 分子安定性のチェック

分子全体が安定かどうかをチェックします。分子が安定であるためには、全ての原子が安定である必要があります。


In [ ]:
def check_molecule_stability(positions, atom_types, charges, dataset_info):
    """
    分子レベルの安定性チェック（厳密）
    
    Parameters:
    -----------
    positions : torch.Tensor [n_atoms, 3]
        原子座標
    atom_types : torch.Tensor [n_atoms]
        原子タイプ
    charges : torch.Tensor [n_atoms, 1]
        電荷
    dataset_info : dict
        データセット情報
    
    Returns:
    --------
    is_stable : bool
        分子が安定かどうか
    """
    n_stable, stable_mask = check_atom_stability(
        positions, atom_types, charges, dataset_info
    )
    
    # 全原子が安定の場合のみ、分子を安定とする
    is_stable = (n_stable == len(positions))
    
    return is_stable

print('分子安定性チェック関数を定義しました。')


## 5. RDKit統合による化学的妥当性検証

RDKitを使用して、より詳細な化学的妥当性を検証します。

**検証項目**:
- SMILES変換の成功
- 分子のサニタイゼーション
- 原子価の整合性
- 芳香族性の判定


In [ ]:
def check_rdkit_validity(positions, atom_types):
    """
    RDKitによる化学的妥当性検証（厳密）
    
    Parameters:
    -----------
    positions : np.ndarray [n_atoms, 3]
        原子座標
    atom_types : np.ndarray [n_atoms]
        原子タイプ
    
    Returns:
    --------
    valid : bool
        RDKitで有効な分子か
    smiles : str or None
        SMILES文字列（変換成功時）
    """
    try:
        # 距離行列から結合を推定
        mol = Chem.RWMol()
        
        # 原子を追加
        for atom_type in atom_types:
            atom_symbol = ATOM_DECODER[atom_type]
            atom = Chem.Atom(atom_symbol)
            mol.AddAtom(atom)
        
        # 結合を推定（距離ベース）
        for i in range(len(positions)):
            for j in range(i + 1, len(positions)):
                dist = np.linalg.norm(positions[i] - positions[j])
                
                # 簡易的な結合判定（原子タイプに応じた閾値）
                if dist < 1.8:  # 典型的な結合距離
                    mol.AddBond(i, j, Chem.BondType.SINGLE)
        
        # サニタイゼーション
        mol = mol.GetMol()
        Chem.SanitizeMol(mol)
        
        # SMILES変換
        smiles = Chem.MolToSmiles(mol)
        
        return True, smiles
    
    except Exception as e:
        # RDKit処理に失敗した場合は無効
        return False, None

print('RDKit妥当性チェック関数を定義しました。')


## 6. 一意性評価（SMILES重複除去）

生成された分子群の一意性を評価します。SMILESの正規化を使用します。

**手法**:
1. 各分子をSMILESに変換
2. Canonical SMILESに正規化
3. 重複を除去
4. 一意性率を計算


In [ ]:
def calculate_uniqueness(smiles_list):
    """
    SMILES一意性の計算（厳密）
    
    Parameters:
    -----------
    smiles_list : list of str
        SMILES文字列のリスト
    
    Returns:
    --------
    uniqueness : float
        一意性率 [0, 1]
    unique_smiles : set
        一意なSMILESの集合
    """
    if len(smiles_list) == 0:
        return 0.0, set()
    
    # Canonical SMILESに正規化
    canonical_smiles = []
    for smiles in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                canonical = Chem.MolToSmiles(mol, canonical=True)
                canonical_smiles.append(canonical)
        except:
            # 変換に失敗した場合はスキップ
            continue
    
    # 一意性の計算
    unique_smiles = set(canonical_smiles)
    uniqueness = len(unique_smiles) / len(smiles_list)
    
    return uniqueness, unique_smiles

print('一意性計算関数を定義しました。')


## 7. 新規性評価

生成された分子が訓練データに含まれていないかを評価します。

**新規性の定義**:
- 訓練データセットに存在しないSMILES
- Tanimoto類似度が一定閾値以下


In [ ]:
def calculate_novelty(generated_smiles, training_smiles):
    """
    新規性の計算（厳密）
    
    Parameters:
    -----------
    generated_smiles : list of str
        生成されたSMILES
    training_smiles : set of str
        訓練データのSMILES集合
    
    Returns:
    --------
    novelty : float
        新規性率 [0, 1]
    novel_smiles : list of str
        新規なSMILESのリスト
    """
    if len(generated_smiles) == 0:
        return 0.0, []
    
    novel_smiles = []
    
    for smiles in generated_smiles:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                canonical = Chem.MolToSmiles(mol, canonical=True)
                
                # 訓練データに含まれていない場合は新規
                if canonical not in training_smiles:
                    novel_smiles.append(canonical)
        except:
            # 変換に失敗した場合はスキップ
            continue
    
    novelty = len(novel_smiles) / len(generated_smiles)
    
    return novelty, novel_smiles

print('新規性計算関数を定義しました。')


## 8. プロパティ分布解析

生成された分子の性質分布を解析します。

**解析する性質**:
1. 分子量
2. logP（脂溶性）
3. 原子数
4. 結合数


In [ ]:
def analyze_property_distribution(smiles_list):
    """
    プロパティ分布の解析（厳密）
    
    Parameters:
    -----------
    smiles_list : list of str
        SMILES文字列のリスト
    
    Returns:
    --------
    properties : dict
        各性質の統計情報
    """
    molecular_weights = []
    logps = []
    n_atoms_list = []
    n_bonds_list = []
    
    for smiles in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue
            
            # 分子量
            mw = Descriptors.MolWt(mol)
            molecular_weights.append(mw)
            
            # logP
            logp = Descriptors.MolLogP(mol)
            logps.append(logp)
            
            # 原子数（水素を含む）
            mol_h = Chem.AddHs(mol)
            n_atoms = mol_h.GetNumAtoms()
            n_atoms_list.append(n_atoms)
            
            # 結合数
            n_bonds = mol.GetNumBonds()
            n_bonds_list.append(n_bonds)
        
        except:
            # エラーは無視
            continue
    
    # 統計情報の計算
    properties = {}
    
    if len(molecular_weights) > 0:
        properties['molecular_weight'] = {
            'mean': np.mean(molecular_weights),
            'std': np.std(molecular_weights),
            'min': np.min(molecular_weights),
            'max': np.max(molecular_weights),
            'values': molecular_weights
        }
    
    if len(logps) > 0:
        properties['logP'] = {
            'mean': np.mean(logps),
            'std': np.std(logps),
            'min': np.min(logps),
            'max': np.max(logps),
            'values': logps
        }
    
    if len(n_atoms_list) > 0:
        properties['n_atoms'] = {
            'mean': np.mean(n_atoms_list),
            'std': np.std(n_atoms_list),
            'min': np.min(n_atoms_list),
            'max': np.max(n_atoms_list),
            'values': n_atoms_list
        }
    
    if len(n_bonds_list) > 0:
        properties['n_bonds'] = {
            'mean': np.mean(n_bonds_list),
            'std': np.std(n_bonds_list),
            'min': np.min(n_bonds_list),
            'max': np.max(n_bonds_list),
            'values': n_bonds_list
        }
    
    return properties

print('プロパティ分布解析関数を定義しました。')


## 9. プロパティ分布の可視化

4種類の性質のヒストグラムを作成します。


In [ ]:
def visualize_property_distributions(properties):
    """
    プロパティ分布の可視化
    
    Parameters:
    -----------
    properties : dict
        プロパティ統計情報
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    prop_names = ['molecular_weight', 'logP', 'n_atoms', 'n_bonds']
    titles = ['分子量', 'logP', '原子数', '結合数']
    
    for i, (prop_name, title) in enumerate(zip(prop_names, titles)):
        ax = axes[i]
        
        if prop_name in properties:
            values = properties[prop_name]['values']
            mean = properties[prop_name]['mean']
            std = properties[prop_name]['std']
            
            ax.hist(values, bins=30, alpha=0.7, edgecolor='black')
            ax.axvline(mean, color='red', linestyle='--', linewidth=2, label=f'平均: {mean:.2f}')
            ax.set_xlabel(title)
            ax.set_ylabel('頻度')
            ax.set_title(f'{title}の分布 (σ={std:.2f})')
            ax.legend()
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, 'データなし', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title)
    
    plt.tight_layout()
    plt.show()

print('可視化関数を定義しました。')


## 10. 包括的評価パイプライン

全ての評価指標を統合した包括的なパイプラインです。


In [ ]:
def comprehensive_evaluation(generated_molecules, dataset_info, training_smiles=None):
    """
    包括的評価パイプライン（厳密）
    
    Parameters:
    -----------
    generated_molecules : list of dict
        生成された分子のリスト
        各要素: {'positions': tensor, 'atom_types': tensor, 'charges': tensor}
    dataset_info : dict
        データセット情報
    training_smiles : set of str, optional
        訓練データのSMILES集合
    
    Returns:
    --------
    results : dict
        評価結果
    """
    n_molecules = len(generated_molecules)
    
    # 初期化
    n_atom_stable = 0
    n_molecule_stable = 0
    n_rdkit_valid = 0
    total_atoms = 0
    valid_smiles = []
    
    # 各分子を評価
    for mol_dict in generated_molecules:
        positions = mol_dict['positions']
        atom_types = mol_dict['atom_types']
        charges = mol_dict['charges']
        
        n_atoms = len(positions)
        total_atoms += n_atoms
        
        # 原子安定性
        n_stable, _ = check_atom_stability(positions, atom_types, charges, dataset_info)
        n_atom_stable += n_stable
        
        # 分子安定性
        is_stable = check_molecule_stability(positions, atom_types, charges, dataset_info)
        if is_stable:
            n_molecule_stable += 1
        
        # RDKit妥当性
        is_valid, smiles = check_rdkit_validity(
            positions.cpu().numpy(), atom_types.cpu().numpy()
        )
        if is_valid:
            n_rdkit_valid += 1
            valid_smiles.append(smiles)
    
    # 一意性
    uniqueness, unique_smiles = calculate_uniqueness(valid_smiles)
    
    # 新規性
    novelty = 0.0
    if training_smiles is not None:
        novelty, _ = calculate_novelty(valid_smiles, training_smiles)
    
    # プロパティ分布
    properties = analyze_property_distribution(valid_smiles)
    
    # 結果の集約
    results = {
        'n_molecules': n_molecules,
        'atom_stability': n_atom_stable / total_atoms if total_atoms > 0 else 0.0,
        'molecule_stability': n_molecule_stable / n_molecules,
        'rdkit_validity': n_rdkit_valid / n_molecules,
        'uniqueness': uniqueness,
        'novelty': novelty,
        'properties': properties,
        'valid_smiles': valid_smiles,
        'unique_smiles': unique_smiles
    }
    
    return results

print('包括的評価パイプラインを定義しました。')


## 11. 品質ベンチマーク

評価結果を品質レベルに分類します。

**品質基準**:

| レベル | 分子安定性 | 一意性 | 新規性 |
|--------|-----------|--------|--------|
| 優秀   | ≥90%      | ≥90%   | ≥70%   |
| 良好   | ≥70%      | ≥70%   | ≥50%   |
| 要改善 | <70%      | <70%   | <50%   |


In [ ]:
def classify_quality(results):
    """
    品質レベルの分類
    
    Parameters:
    -----------
    results : dict
        評価結果
    
    Returns:
    --------
    quality_level : str
        品質レベル（'優秀', '良好', '要改善'）
    """
    mol_stability = results['molecule_stability']
    uniqueness = results['uniqueness']
    novelty = results['novelty']
    
    # 優秀
    if mol_stability >= 0.9 and uniqueness >= 0.9 and novelty >= 0.7:
        return '優秀'
    
    # 良好
    elif mol_stability >= 0.7 and uniqueness >= 0.7 and novelty >= 0.5:
        return '良好'
    
    # 要改善
    else:
        return '要改善'

print('品質分類関数を定義しました。')


## 12. 評価結果の表示

評価結果を見やすく表示します。


In [ ]:
def print_evaluation_results(results):
    """
    評価結果の表示
    
    Parameters:
    -----------
    results : dict
        評価結果
    """
    print('=' * 80)
    print('包括的評価結果')
    print('=' * 80)
    print(f'\n生成分子数: {results["n_molecules"]}')
    print(f'\n安定性メトリクス:')
    print(f'  原子レベル安定性: {results["atom_stability"]*100:.2f}%')
    print(f'  分子レベル安定性: {results["molecule_stability"]*100:.2f}%')
    print(f'  RDKit妥当性: {results["rdkit_validity"]*100:.2f}%')
    print(f'\n多様性メトリクス:')
    print(f'  一意性: {results["uniqueness"]*100:.2f}%')
    print(f'  新規性: {results["novelty"]*100:.2f}%')
    
    if 'properties' in results and len(results['properties']) > 0:
        print(f'\nプロパティ統計:')
        for prop_name, prop_stats in results['properties'].items():
            print(f'  {prop_name}:')
            print(f'    平均: {prop_stats["mean"]:.2f}')
            print(f'    標準偏差: {prop_stats["std"]:.2f}')
            print(f'    範囲: [{prop_stats["min"]:.2f}, {prop_stats["max"]:.2f}]')
    
    quality = classify_quality(results)
    print(f'\n総合品質レベル: {quality}')
    print('=' * 80)

print('結果表示関数を定義しました。')


## 13. コマンドライン実行例

実際のモデル評価は、以下のスクリプトを使用します。


In [ ]:
print('実際の評価コマンド例:')
print('\n# 基本的な評価')
print('python eval_analyze.py \\')
print('    --model_path outputs/qm9_model \\')
print('    --n_samples 10000')
print('\n# 条件付き生成の評価')
print('python eval_conditional_qm9.py \\')
print('    --model_path outputs/conditional_model \\')
print('    --property alpha \\')
print('    --target_values 70.0 75.0 80.0')
print('\n# テストデータでの評価')
print('python eval_sample.py \\')
print('    --model_path outputs/model \\')
print('    --batch_size 100 \\')
print('    --n_samples 1000')


## まとめ

このチュートリアルでは、以下を学習しました:

✅ 安定性メトリクスの詳細計算（原子・分子レベル）  
✅ 許容結合数の厳密な適用  
✅ RDKit統合による化学的妥当性検証  
✅ SMILES重複除去による一意性評価  
✅ 新規性アルゴリズム  
✅ プロパティ分布解析（4種類の性質）  
✅ 包括的評価パイプライン  
✅ 品質ベンチマーク（優秀/良好/要改善）  

### 次のステップ

- **チュートリアル6**: 高度な結晶条件付け - 空間群と密度制御
- **実践**: 実際のモデルで10,000分子を生成し評価
- **発展**: カスタム評価指標の実装

### 重要な原則（再確認）

1. **厳密な検証**: 全ての計算は数学的に厳密
2. **フォールバックなし**: エラーは明示的に報告
3. **再現性**: 評価手法を完全に文書化
4. **透明性**: 品質基準を明確に定義

### 実行例

```bash
# モデルの学習
python main_qm9.py --exp_name qm9_test --n_epochs 500

# 評価の実行
python eval_analyze.py \
    --model_path outputs/qm9_test \
    --n_samples 10000 \
    --save_results results.json
```

---

**質問やフィードバックは、GitHubのIssuesでお寄せください。**
